In [19]:
import pandas as pd
import entsoe
import cdsapi
from dotenv import load_dotenv
import os
import time

_ = load_dotenv()

# Data Acquisition
In our first step we need to gather the data we will be working with.
Please execute the cell above this. 
It will import the needed packages and load your API keys into your environment.

----

We will be using two major data sources: the ENTSO-E Transparency Platform [1] for all data regarding the enrgy markets and the ERA5/Copernicus Dataset [2] for weather data. These datasets are quite extensive, accurate and easily acquired.
We will download the data and store them in a file to be processed and analysed in later steps.

## ENTSO-E

We are starting with the ENTSO-E Datasets as they are the ones we are primarily trying to analyze.
Since I am based in germany we will only be using the data of germany.
However the principals laid out in this project should be adaptable to most other european countries.
We will only be using data from the years 2017 until 2025. This includes major market disruptions due to the ukraine war and the COVID-19 Pandemic.
The Datasets we will be using are:
  - Day-Ahead Prices
  - Actual Total Load
  - Aggregated Generation per Type

The reason to choose these is that energy markets are using the _Merit-Order-Model_ [3] to choose the energy prices.
This model orders the different generators from cheapest running cost to highest running cost.
That is why the Aggregated Generation per Type is interesting to us.
The model then checks what the cheapest set of generators are which will still cover the demand.
That is why the Actual Total Load is interesting.
Finally the price of energy is determined by the running cost of the most expensive generator needed to cover demand.
All other generators are able to sell their "cheaper" energy at the more expensive price.

In [21]:
client = entsoe.EntsoePandasClient(api_key=os.environ['ENTSOE_API_KEY'])

year = 2017
for i in range(9):
    start = pd.Timestamp(f'{year+i}0101', tz='Europe/Brussels')
    end = pd.Timestamp(f'{year+i+1}0101', tz='Europe/Brussels')
    country_code = 'DE_LU'     #  The bidding zone for Germany is the same as for Luxemburg? Luxembourg? TODO
    df = client.query_day_ahead_prices(country_code, start, end)
    df.to_csv(f'data/GER_Price_{year+i}.csv', sep='\t', encoding='utf-8')
    time.sleep(5)
    df = client.query_generation(country_code=country_code, start=start, end=end, nett=False)
    df.to_csv(f'data/GER_Generation_{year+i}.csv', sep='\t', encoding='utf-8')
    time.sleep(5)    
    df = client.query_load(country_code=country_code, start=start, end=end)
    df.to_csv(f'data/GER_Load_{year+i}.csv', sep='\t', encoding='utf-8')
    time.sleep(5)

Connection Error, retrying in 10 seconds


KeyboardInterrupt: 

## Copernicus
Weather data is hugely important for the energy markets,
it directly influences both sides of the Merit-Order-Model: the generation capacity of wind, solar and water energy directly correspond to the weather you are having (or had).
But also the energy consumption changes dramatically with the weather. If it is cold people will be consuming more energy to heat.
While on particularly hot days people might be more prone to turning on air conditioning.

In particularly extreme cases weather can even produce outages and disrupt the entire energy network.

_Note: I am unsure of how much industrial energy consumption varies with the weather._


In [24]:
client = cdsapi.Client(url=os.environ['CDS_API_URL'], key= os.environ['CDS_API_KEY'])


Recovering from connection error [HTTPSConnectionPool(host='cds.climate.copernicus.eu', port=443): Max retries exceeded with url: /api/catalogue/v1/messages (Caused by NameResolutionError("HTTPSConnection(host='cds.climate.copernicus.eu', port=443): Failed to resolve 'cds.climate.copernicus.eu' ([Errno 8] nodename nor servname provided, or not known)"))], attempt 1 of 500
Retrying in 120 seconds


KeyboardInterrupt: 

## Sources
[1] ENTSO-E, "ENTSO-E Transparency Platform," 2026. [Online]. Available: https://transparency.entsoe.eu/. [Accessed: 07.09.2026].
  
[2] H. Hersbach et al., "ERA5 hourly data on single levels from 1940 to present",
    Copernicus Climate Change Service (C3S) Climate Data Store (CDS), 2018.
    [Online]. Available: https://doi.org/10.24381/cds.adbb2d47.
    [Accessed: 07.09.2026].
    
[3] F. Sensfuß, M. Ragwitz, and M. Genoese, "The merit-order effect: A detailed
    analysis of the price effect of renewable electricity generation on spot
    market prices in Germany," Energy Policy, vol. 36, no. 8, pp. 3076–3084, 2008.